# News GNN Ablation — Kaggle (GPU) — TabDiff XGBoost evaluation

Runs `ablation/ablation_runner.py` on the **news** dataset using the **dual-split
design**:

- **This notebook** trains, samples, and evaluates entirely on the dedicated
  TabDiff-comparable split `data/news_tabdiff_split/` (built by
  `scripts/make_tabdiff_splits.py`, seed 0, sizes matching TabDiff's Table 6),
  with **TabDiff's exact XGBoost protocol** (`ablation/eval_xgboost.py`, a
  faithful port of `eval/mle/mle.py` from MinkaiXu/TabDiff). Results land under
  `exp/news_tabdiff_split/` so they are never conflated with the original-split results.
- The **original split** `data/news/` stays untouched — it remains the one used
  for the CatBoost/TabDDPM-protocol comparison, with its own results under
  `exp/news/`.

Downstream reporting keeps these as **two separate tables** (CatBoost vs TabDDPM,
XGBoost vs TabDiff) — never merged into one number.

**How to use**
1. In the notebook sidebar set **Accelerator → GPU** and **Internet → On**.
2. Run the setup cells (clone + install) once.
3. Edit only the **ABLATION PARAMETERS** cell to change what gets swept, then run the last cell.

These parameters mirror the module-level constants in `ablation_runner.py`
(`GNN_TYPES`, `N_LAYERS`, `D_MODELS`, `TOP_KS`, `N_HEADS`, `ATTENTIONS`,
`ABLATION_STEPS`, seed / diffusion constants). We override them in-memory so you
never have to touch the source file.


## 1. Check GPU


In [ ]:
import os
# Bound CUDA fragmentation so the multi-seed sampling loop doesn't OOM on the
# 15 GB T4 (seed 0 fit, seed 1 was OOMing in the GAT/FFN forward). Must be set
# before torch initialises its CUDA context, i.e. before the first cuda call.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))


## 2. Get the repo (+ data)

Default: clone the public GitHub repo (data is committed in the repo, ~0.5 GB).

If the repo is **private**, either:
- upload it as a Kaggle Dataset and set `REPO_SOURCE = 'kaggle'` + `KAGGLE_DATASET_DIR`, or
- put a GitHub token in `GIT_TOKEN` below.


In [2]:
import os, shutil

REPO_SOURCE = 'git'                 # 'git'  or  'kaggle'
GIT_URL     = 'https://github.com/stefromp/thesis_project'
GIT_BRANCH  = 'main'
GIT_TOKEN   = ''                    # optional: personal access token for a private repo

# Only used when REPO_SOURCE == 'kaggle' (repo uploaded as a Kaggle Dataset)
KAGGLE_DATASET_DIR = '/kaggle/input/thesis-project'

REPO_ROOT = '/kaggle/working/thesis_project'

if REPO_SOURCE == 'git':
    if os.path.isdir(REPO_ROOT):
        shutil.rmtree(REPO_ROOT)
    url = GIT_URL
    if GIT_TOKEN:
        url = url.replace('https://', f'https://{GIT_TOKEN}@')
    !git clone --depth 1 --branch {GIT_BRANCH} {url} {REPO_ROOT}
elif REPO_SOURCE == 'kaggle':
    # Kaggle input is read-only, so copy it into the writable working dir.
    if os.path.isdir(REPO_ROOT):
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(KAGGLE_DATASET_DIR, REPO_ROOT)
else:
    raise ValueError('REPO_SOURCE must be "git" or "kaggle"')

# This notebook runs on the dedicated TabDiff split; the original split must
# also be present (untouched) — both ship with the repo.
assert os.path.isdir(os.path.join(REPO_ROOT, 'data', 'news_tabdiff_split')), \
    'TabDiff split not found under REPO_ROOT/data/news_tabdiff_split — run scripts/make_tabdiff_splits.py and commit it'
assert os.path.isdir(os.path.join(REPO_ROOT, 'data', 'news')), \
    'original news data not found under REPO_ROOT/data/news'
print('REPO_ROOT =', REPO_ROOT)
print(sorted(os.listdir(REPO_ROOT)))


Cloning into '/kaggle/working/thesis_project'...
remote: Enumerating objects: 1027, done.
remote: Counting objects: 100% (1027/1027), done.
remote: Compressing objects: 100% (677/677), done.
remote: Total 1027 (delta 321), reused 991 (delta 311), pack-reused 0 (from 0)
Receiving objects: 100% (1027/1027), 1.03 MiB | 6.45 MiB/s, done.
Resolving deltas: 100% (321/321), done.
Filtering content: 100% (326/326), 691.31 MiB | 44.66 MiB/s, done.
REPO_ROOT = /kaggle/working/thesis_project
['.DS_Store', '.git', '.gitattributes', 'abalone-ablation-kaggle.ipynb', 'ablation', 'adult_ablation-kaggle-tabdiff.ipynb', 'catboost_info', 'checkpoints', 'data', 'docs', 'exp', 'lib', 'requirements.txt', 'scripts', 'tab_ddpm', 'tabddpm_ablation', 'tests', 'tuned_models']


## 3. Install dependencies

We keep Kaggle's pre-installed GPU build of **torch / numpy / pandas / scipy /
scikit-learn** (downgrading them tends to break CUDA) and only add the extra
packages the eval pipeline needs. `xgboost` is added for TabDiff's evaluator, `sdmetrics` for the paper's
fidelity metrics (Shape / Trend / C2ST)
(Kaggle preinstalls a GPU-capable build, so this is usually a no-op).

We deliberately do **not** install `rtdl` / `libzero` — they are source-only on
recent Python and fail to build, and `ablation_runner` injects lightweight stubs
for the `zero` / `rtdl` modules automatically. Packages are installed unpinned so
pip can pick a wheel, and each on its own line so one failure can't abort the rest.


In [3]:
import sys, importlib, subprocess

# (pip_name, import_name) for everything the pipeline imports.
DEPS = [
    ('icecream',          'icecream'),
    ('catboost',          'catboost'),
    ('category-encoders', 'category_encoders'),
    ('tomli',             'tomli'),
    ('tomli_w',           'tomli_w'),
    ('pynvml',            'pynvml'),
    ('skorch',            'skorch'),
    ('xgboost',           'xgboost'),
    ('sdmetrics',         'sdmetrics'),
]

def have(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

# Only install what's actually missing (Kaggle preinstalls most of these), and
# force a wheel so pip never falls back to a source build (the build failures).
missing = [(pip, mod) for pip, mod in DEPS if not have(mod)]
print('missing:', [m for _, m in missing] or 'none')

for pip_name, mod in missing:
    print(f'>>> installing {pip_name}')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', pip_name],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        # Fall back to allowing a source build, but show the real error if it fails.
        r2 = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name],
            capture_output=True, text=True,
        )
        if r2.returncode != 0:
            print(f'    !! FAILED {pip_name}\n{(r.stderr or r2.stderr)[-1500:]}')

importlib.invalidate_caches()
print('\nfinal check:')
for _, mod in DEPS:
    print('  ok  ' if have(mod) else '  FAIL', mod)


missing: ['icecream', 'tomli', 'tomli_w', 'skorch', 'sdmetrics']
>>> installing icecream
>>> installing tomli
>>> installing tomli_w
>>> installing skorch
>>> installing sdmetrics

final check:
  ok   icecream
  ok   catboost
  ok   category_encoders
  ok   tomli
  ok   tomli_w
  ok   pynvml
  ok   skorch
  ok   xgboost
  ok   sdmetrics


## 4. News dataset configuration (TabDiff split)

Base config pulled from the repo's registry
(`ablation/run_all_ablation.py` → `ALL_DATASETS['news']`), then repointed at the
dedicated TabDiff split `data/news_tabdiff_split/`. `num_samples` is set to the split's
train size, matching TabDiff's protocol of evaluating on a synthetic table the
same size as the real training set.

The cell **confirms the TabDiff split matches TabDiff's Table 6 sizes exactly** (it was
constructed to match, so this is an assert, not a warning), and states where the
untouched original split lives.

> **Feature schema for this split now matches TabDiff's official preprocessing**:
> the 14 one-hot dummies are collapsed into 2 categorical columns
> (`data_channel` = argmax of the 6 `data_channel_is_*` dummies, `weekday` = argmax
> of `weekday_is_monday..sunday` + `is_weekend`), and `timedelta` is kept as a
> numerical feature (re-attached from the raw CSV, row-alignment verified) —
> 45 num / 2 cat. The original `data/news/` (44 num / 14 dummy cat, used for the
> CatBoost/TabDDPM comparison) is unchanged.


In [ ]:
import sys
import numpy as np

for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'ablation'), os.path.join(REPO_ROOT, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

from run_all_ablation import ALL_DATASETS

DATASET_NAME     = 'news'
TABDIFF_SPLIT    = 'news_tabdiff_split'          # data folder AND exp/ output name
EXPECTED_SIZES   = {'train': 31714, 'val': 3965, 'test': 3965}

CFG = dict(ALL_DATASETS[DATASET_NAME])
CFG['real_data_path'] = f'data/{TABDIFF_SPLIT}/'
CFG['num_samples']    = EXPECTED_SIZES['train']   # TabDiff: synthetic == train size
CFG['num_numerical_features'] = 45  # TabDiff schema: timedelta kept + dummies collapsed to 2 cat cols (see data/news_tabdiff_split/info.json)
CFG['sample_batch_size']      = 10000  # T4 15 GB: smaller sampling batch halves the GAT/FFN peak so the multi-seed loop stays in memory (sampling is inference-only -> changes speed, not results)
print(CFG)

# ---- confirm the TabDiff split matches the expected sizes exactly ----
data_dir = os.path.join(REPO_ROOT, CFG['real_data_path'])
sizes = {s: len(np.load(os.path.join(data_dir, f'y_{s}.npy'), allow_pickle=True))
         for s in ('train', 'val', 'test')}
assert sizes == EXPECTED_SIZES, f'TabDiff split size mismatch: {sizes} != {EXPECTED_SIZES}'
print('TabDiff split verified:', sizes)
print('Original split (CatBoost/TabDDPM protocol) is untouched at',
      os.path.join(REPO_ROOT, 'data', DATASET_NAME),
      '— its results live under exp/' + DATASET_NAME + '/, separate from this run.')


## 5. ABLATION PARAMETERS  ← edit this cell each run

These are exactly the constants you'd otherwise edit at the top of
`ablation_runner.py`. The full grid is the Cartesian product of the lists
(GCN/GIN ignore `N_HEADS`, so only the first head value runs for them).

Grid size = `len(GNN_TYPES) * len(N_LAYERS) * len(D_MODELS) * len(TOP_KS) * len(N_HEADS) * len(ATTENTIONS)`
(minus the redundant head combos for gcn/gin).


In [5]:
# ---- ablation grid (change these) ----
GNN_TYPES  = ['gat']   # subset of: gcn | gat | gatv2 | gin
N_LAYERS   = [3]                   # number of blocks
D_MODELS   = [128]                # hidden dim
TOP_KS     = [5]                  # sparsity_top_k (0 = dense)
N_HEADS    = [8]                     # attention heads (gcn/gin ignore)
ATTENTIONS = [False]                 # dense self-attention sublayer on/off

# ---- training / eval budget ----
ABLATION_STEPS         = 20_000
N_GEN_SEEDS            = 5
# TabDiff's XGBoost protocol is deterministic (fixed grid, no per-seed
# randomness once the val split is fixed), so 1 clf seed per gen seed is
# enough — extra seeds would just repeat identical evaluations.
N_CLF_SEEDS            = 1
ABLATION_NUM_TIMESTEPS = 1000
ABLATION_BATCH_SIZE    = 4096
ABLATION_LR            = 1e-4

# ---- run options ----
DEVICE  = 'cuda:0'
SEED    = 0
NO_SKIP = False   # True = re-run combos even if results already exist


## 6. Run the ablation (on the TabDiff split)

Imports the real `ablation_runner`, overwrites its module-level constants with
the values from the cell above, selects the **TabDiff XGBoost evaluator**
(`ar.EVALUATOR = 'xgboost'`) and the **TabDiff paper fidelity suite**
(`ar.TABDIFF_FIDELITY = True`: density Shape/Trend via sdmetrics, C2ST
detection, TabDiff DCR, naive α-precision/β-recall), then trains/samples/evaluates the full grid **on
the TabDiff split** — training data, validation, and test all come from
`data/news_tabdiff_split/`. Results land in
`{REPO_ROOT}/exp/news_tabdiff_split/ablation/<combo>/results_full_averaged.json` and a
consolidated `ablation_summary.json`, fully separate from the original-split
results in `exp/news/`.


In [ ]:
import sys, argparse

for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'ablation'), os.path.join(REPO_ROOT, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

# eval_catboost loads tuned_models/catboost/<ds>_cv.json via a RELATIVE path,
# so the process cwd must be the repo root (harmless for xgboost, kept for parity).
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

import ablation_runner as ar

# Override the module constants with the values chosen above.
ar.GNN_TYPES             = GNN_TYPES
ar.N_LAYERS              = N_LAYERS
ar.D_MODELS              = D_MODELS
ar.TOP_KS                = TOP_KS
ar.N_HEADS               = N_HEADS
ar.ATTENTIONS            = ATTENTIONS
ar.ABLATION_STEPS        = ABLATION_STEPS
ar.N_GEN_SEEDS           = N_GEN_SEEDS
ar.N_CLF_SEEDS           = N_CLF_SEEDS
ar.ABLATION_NUM_TIMESTEPS = ABLATION_NUM_TIMESTEPS
ar.ABLATION_BATCH_SIZE   = ABLATION_BATCH_SIZE
ar.ABLATION_LR           = ABLATION_LR
ar.EVALUATOR             = 'xgboost'   # TabDiff's eval/mle protocol
ar.TABDIFF_FIDELITY      = True        # TabDiff paper suite: Shape/Trend, C2ST, DCR, alpha/beta

combos = list(ar.iter_ablation_grid())
print(f'Total combinations to run: {len(combos)}')
for c in combos:
    print('  ', ar.exp_dir_name(*c))

# ============================================================================
# Reuse the pre-trained model (skip §training) instead of retraining
# ----------------------------------------------------------------------------
# The model was trained in a previous run and saved as a Kaggle Dataset. Copy it
# into the path the runner expects (exp/<split>/ablation/<combo>/model_ema.pt),
# then patch the trainer to a no-op whenever a model is already there — so
# run_ablation jumps straight to sampling + evaluation. Fails loudly if the
# model isn't found, so we never silently burn 20k steps retraining.
import glob, shutil
# train.py does `import zero` / rtdl at module load; those aren't pip-installed
# (run_ablation injects lightweight stubs). Install the stubs now so we can
# import `train` here, before run_ablation does its own (idempotent) setup.
ar._setup_zero_rtdl_stubs()
import train as _train_mod

PRETRAINED_SEARCH_ROOTS = [
    '/kaggle/input/datasets/stef16/news-tabdiff-gat-model',   # exact path of the uploaded model
    '/kaggle/input',                                          # fallback: search every attached dataset
]

_all_ema = sorted({h for root in PRETRAINED_SEARCH_ROOTS
                     for h in glob.glob(os.path.join(root, '**', 'model_ema.pt'), recursive=True)})
print('\nstored model_ema.pt found:')
for h in _all_ema:
    print('   ', h)

for c in combos:
    name    = ar.exp_dir_name(*c)
    dst_dir = os.path.join(REPO_ROOT, 'exp', TABDIFF_SPLIT, 'ablation', name)
    dst_ema = os.path.join(dst_dir, 'model_ema.pt')
    if os.path.exists(dst_ema):
        print(f'[model] already in place: {dst_ema}')
        continue
    # Prefer a stored file whose path names this combo; else, when a single
    # combo runs, take the only stored model.
    match = [h for h in _all_ema if name in h] or (_all_ema if len(combos) == 1 else [])
    if not match:
        raise FileNotFoundError(
            f'No stored model_ema.pt for combo {name}. Attach the Kaggle Dataset '
            f'holding your trained model, or edit PRETRAINED_SEARCH_ROOTS '
            f'(searched: {PRETRAINED_SEARCH_ROOTS}).')
    src_dir = os.path.dirname(match[0])
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy(match[0], dst_ema)                      # sampler loads this
    src_pt = os.path.join(src_dir, 'model.pt')
    if os.path.exists(src_pt):
        shutil.copy(src_pt, os.path.join(dst_dir, 'model.pt'))
    print(f'[model] restored {src_dir} -> {dst_dir}')

# Patch the trainer: no-op when a model is already on disk for that combo.
if not getattr(_train_mod.train, '_skip_wrapper', False):
    _real_train = _train_mod.train
    def _train_or_skip(*a, **k):
        pdir = k.get('parent_dir')
        ema  = os.path.join(pdir, 'model_ema.pt') if pdir else None
        if ema and os.path.exists(ema):
            print(f'[skip-train] reusing existing model -> {ema}')
            return
        return _real_train(*a, **k)
    _train_or_skip._skip_wrapper = True
    _train_mod.train = _train_or_skip
# ============================================================================

args = argparse.Namespace(
    repo_root=REPO_ROOT,
    data_root=REPO_ROOT,
    exp_root=REPO_ROOT,
    device=DEVICE,
    seed=SEED,
    skip_if_done=True,
    no_skip=NO_SKIP,
    aggregate=False,
)

# Passing TABDIFF_SPLIT as the dataset name routes every output (models,
# samples, metrics, summary) to exp/news_tabdiff_split/ — never exp/news/.
summary = ar.run_ablation(TABDIFF_SPLIT, CFG, args, single_combo=None)


## 7. Inspect / save results

Everything under `exp/news_tabdiff_split/ablation/` is written into `/kaggle/working`, so
it is downloadable from the notebook's **Output** tab after the session ends.
ML-efficiency columns are TabDiff-style (R2 / RMSE (log-scale)); DCR / Wasserstein /
MIA-AUC are the TabDDPM-protocol metrics, unchanged.
Shape / Trend / C2ST / DCR / aPrec / bRec are the TabDiff-paper suite
(all averaged over generation seeds; DCR ~0.5 is ideal, the rest higher = better). This table is the
**TabDiff-split / XGBoost** result — keep it separate from the original-split
CatBoost table under `exp/news/`.


In [7]:
import json

summary_path = os.path.join(REPO_ROOT, 'exp', TABDIFF_SPLIT, 'ablation', 'ablation_summary.json')
with open(summary_path) as fh:
    data = json.load(fh)

print('summary:', summary_path, '\n')

def g(metrics, key):
    e = metrics.get(key) if metrics else None
    return f"{e['mean']:.4f}" if isinstance(e, dict) and 'mean' in e else '  -  '

# ML efficiency via TabDiff's XGBoost protocol (best-per-metric on val, scored
# on the TabDiff split's own test partition); plus the TabDDPM privacy metrics.
hdr = (f"{'combo':40s} {'status':8s} {'R2':>10s} {'RMSE(log)':>10s} "
       f"{'Shape':>7s} {'Trend':>7s} {'C2ST':>7s} {'DCR':>7s} "
       f"{'aPrec':>7s} {'bRec':>7s} "
       f"{'DCR_mean':>10s} {'Wass':>8s} {'MIA_AUC':>8s}")
print(hdr); print('-' * len(hdr))
for e in data:
    m = e.get('metrics', {})
    print(f"{e['exp_name']:40s} {e.get('status',''):8s} "
          f"{g(m,'r2'):>10s} {g(m,'rmse'):>10s} "
          f"{g(m,'shape'):>7s} {g(m,'trend'):>7s} {g(m,'c2st'):>7s} {g(m,'dcr_tabdiff'):>7s} "
          f"{g(m,'alpha_precision'):>7s} {g(m,'beta_recall'):>7s} "
          f"{g(m,'dcr_tabddpm_mean'):>10s} {g(m,'wasserstein_mean'):>8s} "
          f"{g(m,'mia_auc'):>8s}")


summary: /kaggle/working/thesis_project/exp/news_tabdiff_split/ablation/ablation_summary.json 

combo                                    status           R2  RMSE(log)   Shape   Trend    C2ST     DCR   aPrec    bRec   DCR_mean     Wass  MIA_AUC
----------------------------------------------------------------------------------------------------------------------------------------------------
gnn_gat_layers3_dim128_topk5_heads8_noattn failed          -          -       -       -       -       -       -       -          -        -        -  
